In [55]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt


In [56]:
df = pd.read_csv('train.txt', sep = ";", header = None, names = ['text', 'emotion'])

In [57]:
df.head()

,text,emotion
0,i didnt feel humiliated,sadness
1,i can go from feeling so hopeless to so damned...,sadness
2,im grabbing a minute to post i feel greedy wrong,anger
3,i am ever feeling nostalgic about the fireplac...,love
4,i am feeling grouchy,anger


In [58]:
df.isnull().sum()

text       0
emotion    0
dtype: int64

In [59]:
unique_emotions = df['emotion'].unique()
emotion_num = {}
i = 0
for emo in unique_emotions:
  emotion_num[emo] = i
  i += 1

df['emotion'] = df['emotion'].map(emotion_num)


In [60]:
df

,text,emotion
0,i didnt feel humiliated,0
1,i can go from feeling so hopeless to so damned...,0
2,im grabbing a minute to post i feel greedy wrong,1
3,i am ever feeling nostalgic about the fireplac...,2
4,i am feeling grouchy,1
...,...,...
15995,i just had a very brief time in the beanbag an...,0
15996,i am now turning and i feel pathetic that i am...,0
15997,i feel strong and good overall,5
15998,i feel like this was such a rude comment and i...,1


In [61]:
df['text'] = df['text'].apply(lambda x:x.lower())

In [62]:

import string
def remove_punc(txt):
  return txt.translate(str.maketrans('', '', string.punctuation))

In [63]:
df['text'] = df['text'].apply(remove_punc)

In [64]:
def remove_numbers(txt):
  new = ""
  for i in txt:
    if not i.isdigit():
      new = new + i
  return new
df['text'] = df['text'].apply(remove_numbers)

In [65]:
def remove_emojis(txt):
  new = ""
  for i in txt:
    if i.isascii():
      new += i
  return new
df['text'] = df['text'].apply(remove_emojis)

In [66]:
import nltk

In [67]:
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
nltk.download('stopwords')
stop_words = set(stopwords.words('english'))

keep_words = ['not', 'no', 'never']

for word in keep_words:
    stop_words.discard(word)


[nltk_data] Downloading package stopwords to C:\Users\KOMAL
[nltk_data]     ANAND\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [68]:
stop_words

{'a',
 'about',
 'above',
 'after',
 'again',
 'against',
 'ain',
 'all',
 'am',
 'an',
 'and',
 'any',
 'are',
 'aren',
 "aren't",
 'as',
 'at',
 'be',
 'because',
 'been',
 'before',
 'being',
 'below',
 'between',
 'both',
 'but',
 'by',
 'can',
 'couldn',
 "couldn't",
 'd',
 'did',
 'didn',
 "didn't",
 'do',
 'does',
 'doesn',
 "doesn't",
 'doing',
 'don',
 "don't",
 'down',
 'during',
 'each',
 'few',
 'for',
 'from',
 'further',
 'had',
 'hadn',
 "hadn't",
 'has',
 'hasn',
 "hasn't",
 'have',
 'haven',
 "haven't",
 'having',
 'he',
 "he'd",
 "he'll",
 "he's",
 'her',
 'here',
 'hers',
 'herself',
 'him',
 'himself',
 'his',
 'how',
 'i',
 "i'd",
 "i'll",
 "i'm",
 "i've",
 'if',
 'in',
 'into',
 'is',
 'isn',
 "isn't",
 'it',
 "it'd",
 "it'll",
 "it's",
 'its',
 'itself',
 'just',
 'll',
 'm',
 'ma',
 'me',
 'mightn',
 "mightn't",
 'more',
 'most',
 'mustn',
 "mustn't",
 'my',
 'myself',
 'needn',
 "needn't",
 'nor',
 'now',
 'o',
 'of',
 'off',
 'on',
 'once',
 'only',
 'or',
 'o

In [69]:
def remove(txt):
  words = word_tokenize(txt)
  cleaned  = []
  for word in words:
    if word not in stop_words:
      cleaned.append(word)
  return " ".join(cleaned)

In [70]:
nltk.download('punkt_tab')
df['text'] = df['text'].apply(remove)

[nltk_data] Downloading package punkt_tab to C:\Users\KOMAL
[nltk_data]     ANAND\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [71]:
df.loc[1]['text']

'go feeling hopeless damned hopeful around someone cares awake'

In [72]:
from sklearn.model_selection import train_test_split
X_train, X_test,y_train,y_test =train_test_split(df['text'],df['emotion'],test_size = 0.20, random_state=42)

In [73]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer


In [74]:
tfidf_vectorizer = TfidfVectorizer(
    max_features=15000,
    ngram_range=(1,3),
    min_df=2,
    max_df=0.90,
    sublinear_tf=True
)
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)


In [75]:
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score

svm_model = LinearSVC(
    C=2,
    max_iter=8000
)

svm_model.fit(X_train_tfidf, y_train)

svm_pred = svm_model.predict(X_test_tfidf)

print(
    "SVM Accuracy:",
    accuracy_score(y_test, svm_pred)
)

SVM Accuracy: 0.9021875


In [76]:
import pickle

In [77]:
# Save SVM model
pickle.dump(
    svm_model,
    open('svm_model.pkl', 'wb')
)

In [78]:
# Save TF-IDF vectorizer
pickle.dump(
    tfidf_vectorizer,
    open('tfidf_vectorizer.pkl', 'wb')
)

In [54]:
# Save emotion labels
pickle.dump(
    emotion_num,
    open('emotion_labels.pkl', 'wb')
)